## 1. Detect zipped or automatically unzipped input


In [ ]:
from pathlib import Path
import base64, json, os, subprocess, sys, time, zipfile
INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
RUN_ROOT = WORK_ROOT / "jcam_k2_fixed_regularity_upper"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_OK = True
STAGES = []
def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)
mode = "embedded_exact_sources"
if INPUT_ROOT.exists():
    if any(INPUT_ROOT.rglob("check_k2_coverage_identity.py")):
        mode = "kaggle_auto_unzipped_detected; embedded exact sources selected"
    elif any(INPUT_ROOT.rglob("*.zip")):
        mode = "kaggle_zip_detected; embedded exact sources selected"
record("bootstrap", "passed", mode)


## 2. Install exact source package from notebook cell


In [ ]:
SOURCE_ARCHIVE = base64.b64decode("".join([
    'UEsDBBQAAAAIAJE1Q13rES5COwQAAC4KAAAuAAAAQ29kZS9leHBlcmltZW50cy9jaGVja19rMl9jb3ZlcmFnZV9pZGVudGl0',
    'eS5weZVW227jNhB911cM9ESmshIbSNG66wILd5+6aArvbvtgGAIj0REbSRRIKtam2H/vDClpbccJUAMGb3PmeoZUHMd/dLU0',
    'KhcV5KXMHy3stQFXSmiNfpIFuIOetVo1DnJcG/EgQRWyccp9BdEU0LWtNHCvu6ZI4ziOor3RNWTZvnOdkVkGqm61cSjbaCec',
    '0o2NomEvt0/j9B+rmwBthSsrdT/i/sRlFEWbu7vPsPIrhspVhap5aqTV1ZNkPG2FQafsdrGL7r6QpAdcQ7zWhYxpgrJd5ayf',
    '71Uvi+xxkY0xxYRK68dCGTaoWn02nUxA9sq6TD/6JY/WqPomvY02fpwvbqNP2V/vP3758Ak32E16czNPgIZFGG79EPbmtzy6',
    '27xff/yAsj/DFWz8/xrmP2KEhdyDE+ZBuqzS1rJ+CftKC5dAN82wLk4MKw6zX8NsGQH+jMSEN1CLnvk1/dBoMi1YD7OgAX6A',
    'DYerK1jgDutebl8EzS6DZqcgPoQie5G7rBU+ocoswXVtJbdDIH7YvR1Qj1nrF5gqizyQhVfD/UmdQEnp7ufocr/gmEJMN+vJ',
    's37ul17uGYXq0VG/I6yVRMZ7y545gkt4h1TByVzO5ovzPLINgks8DfJDmMAWQ+1KrwFHPAE2R+FnnDyjfeZLyzllw+iDRUe2',
    'u4hay4JqYORMCNTHQk7YIb/XwcRgh/ZD3JQBkl2TWwmsyTw/joscsj6kEIq+76xDxFkxEBpgsilCb18S8YbPszaonE1QDu+O',
    'kof9I1GNNEabIWjaHu4UZAsG7923aGC0sZzYNpWcXJh2qYzrkzLSLzfYJap5wMORvc+hDFSUa8z+JBraivxhE2eTSUFyxP4J',
    'EgIh/iAPatWwk9ZEN7sBxpPTrl0cHXEfeEdBDy6cGxgylQq8RZuCUYYHy7OXFRmUntSE3DtRdVoP4t6ofDL97zSjX2zjJZbj',
    'dK8U1T57EC0elWdH2ogcb198FzJPgAxbVhsUDFfbmXTgy3TPZv65yOhv6MlAWJC4DJN9Lq19FYRJumj0BEaWs4MqXIk4dgbk',
    'vtfSWySO5WdaMLXZ+NSF5KKCl/n+jvo2XoAH5Upg9A5demtSfPZinmosCosPcQKNPFSqkasY57LJdYGkXMWd289+ijkWGui5',
    'C01yMMpJugNQR/qbyt3ffoORBN6qSlZFI2ppVxW+Woyqv73ZDYwJ2NQPpRQFwl4eEMTjeGS7uhbmKxoLhIlzYSWRpULHvUgI',
    'PRZVlYWPB2Qr0pIK5B/P6O08oo7tpdOdbxs8pcY5NhQKOlT3e10zIxp8wpewnUpBHevVv0aF12x48Ojb/wXvBj/xS4fcie+a',
    'ij6SQOSlkvQxdfSx9AsI+H21IE8VGoRKH8YjUBaIHWmcRN+iV3mUDQVK6dsJCeXrlznZO0Y7adHVrWWDUIKuUpZXC36BZFFr',
    'sJPfhvHoP1BLAwQUAAAACAAINUNdSPMsBLYCAADBBQAAJAAAAENvZGUvZXhwZXJpbWVudHMvZXhwbG9yZV9maXhlZF9rMi5w',
    'eW1T226jMBB95ytGPEEKJLCbVo2Wfan2B6K+VRWiMCReAUa+pDRR/n3HNrl2UYIN58yZM+PB9/0/49ByUSouvoAPinVsXyrG',
    'e2i4APXJY67VoBU0bMQ6FrjRbSmY+oKK71CUG0w873XLJNCvhF53KFhVtsD6HUrFNlYsgp4rgtUWucAOSLpCoVjDsD5lRZF4',
    'vu97XiN4B0XRaKUFFgWwbuCConvSsGpy4siKDV/JKfzEq1nToMBesbItcMdbbWI8z3uBHBbJ0lvbNc2W9K7G5lxIMZRMBOa2',
    'AqWHFt+alpcqAru8R8a9KlfuMYT4t9utPKBrTCMYM1KW5AFrKxNaZE/I3iBjCrHTsNRpbzmllNQOiNfwK6cAe8/MfW3hFhtF',
    'AgEhD7AOYTaDzAKCbbYWWZPcPrtCKsGlZP3GgIQ+nFEiOnbqnkOYQ/ATZqRrAztW1y2eNdPbqJPstQekc+qhK8fA+IwmhciZ',
    'C6nJNF6VlXw2aex/DumjZyaM5qaHYJEsFtQms2RuWdrFvctC1+NPLtpaGmsvZEVG8EJ1SWf7g+veYRY0GTJDiMn7jEiRCXpw',
    'Dy7QUkIXbZx0vEZrxu+w7P0IfMonlT8lN5eZFv7xFyvFduiO+AKaa1e2Go2Lt+9DNc1P6D4sszXZXE3vNypTR6XuAidozigD',
    '1jiPeQ7OImAr0TZ+onneRULq1kzG/7+G4FxFNHWO3PE2TzFOqW0Db5nc5q9CEy4R6zx9Cs/a96PuciXjhVFxytdrrmVhe0hc',
    '4/KmyNsGBU40dEcmTydFf2ZmZbFYhDfRpofM9E+U/QYDIqQXwmU3CNar27yNL/ODXCU/Nkc4mH4eYcyDw5iukqfmGB3GzG5C',
    '8O/CcKxQUuxUbqP72A32KnlujuDgORGCb4xwTgkfiXSved+o4pTkHrhO5V8V+g9QSwMEFAAAAAgAfTVDXW+SUf7aBgAAYQ0A',
    'ACkAAABNYXRoL3Byb29mcy9wcm9vZl9rMl9jb3ZlcmFnZV9pZGVudGl0eS5tZH1Xy27bRhTd6ysGyKKUJVqWnARNUgZI2wQJ',
    'kjZB7bYLIzFH5NCchuQoM0NZfm2LdF2g+wLpDxgo4LU/wB/hL+m5M0NJjoMuEsgczn2ee87lHfbq6Q/x1oSN2JufXr/Br4dM',
    'LHhmmT1U8UzJxrIPLc81tzJjmZoLzQ8E403OONNq2hrL2tlMaDZVbZP3ehsbO5bb1jwke788/Z4VSjNbCrb7HH4eMKt5Yypu',
    'Rb5mt+C1rI42NzbYbikN08K0lWW5EoY1yjJhLJ9W0pTwWciFyGMtDtqKa2mPWPoymaSslo2s+YJV6rCLZbPXu3OHPXXJLAOX',
    'uWgsrvV6z+SCqUawQ6WrnKVXF+mQiaKQmcQbDHkLPecVS5/uX10ke1cXsR5eXQz0W7xG2aM8TOlcaCSiWjtrLXPVMixd7I+T',
    'Oi6v//i02J8k9aBMYc4bwuVDaUuW4vSfrdRZSk/r+OridEAXdLqJEoim10uv/sX70QmMDWHmrM8ShgRPWKTjcuCv9C/Phyya',
    '6HJQXp7398ZxRI/xdKQvz9+yMxiLXk7icb/Xg1GUTqNbyKv29TCuL8jfiLXMRZO7RFwgzIhMIUbp312WsVLGMG79Q40/ZHPA',
    'LNcHwrKpsIdCNO7MVclVx2yyFwXyRdF4VpFhy0puuqLhJYPeLKuUVXI2c0ZLUVM3VDhiGW8IE7LJtOBmFRNwy7OSEbSQhZr+',
    'JjIr54Llwr9o2CEKy2o1J7OFVvXSqVWHXCNJxNEIDvTZL9bjO9UY8aHFw+rIZTelNx1+YjMTmcSVtbEpKgXoo3Junqoj39PJ',
    'u40IPUoeoEWj8X1Ck7VcNoDRFGDeQY7vNpITwtvorkPc6G7XyEmf5uuNVqqgWfkV+EfFjhPqusfSjGteC7RYHmNIu46gzGmb',
    'wNYc7tL59cePe0AzoOx6ki6AxRYnlH+GmHPZoIiHEsUF4mTd1uiSka6arvHIKY0WcduPFoM2nsDwRPfTR0x6Y5+8Mf9aGy/6',
    'gGiM1wd4sw+frz02KlGgT4WbF1ywS4SGcXIjjTwIejDlIUeTV67iUlnWaofEdJ7EGmYOpOtv6sbkGLOwcqjlQdl5nASPAd+3',
    'XCL8//X1uas4uPp2DfwB+OupOWR8FZLpILxKy0eD8yVo3RRKfWvIqLbz/Y3kOBoPyhGqD2IpJeFfwt70dhR4lvEWLU1Pj0+J',
    'aeIS4e60UwM6bK1UDeWDu0tPLsZ0SS/ROD72zEI93OXvw3Ri4hCRcYV1I19qIRzLYLi1IpOehMDHTyzBdWtYJkA1Qp4qcKF/',
    'FYNFbLsaC7pLLlaUsaYzm+xZkBUtiMnEkBkcKyPcw25mPdmhVsZqSTOIKkAhVm4824KOHyOgASqDJ69Q3tQm/o8nxnH1J+p3',
    'WkRlPwn1wLNtGAlqUCtwkmpAGVAkQliRhnK6SxiwIiIPln4G54Ponh5N+hZenqHXcioruh2uWedg9DUcoBsZ+lYObFCI54Ie',
    '3OTekCjQWcGYBdmQYzTNhqYxeGafuY63ccpWIaXo0Y5aGnH66C03oYmoN1hWBDmouKxBXJ7pSOipGUescOmgYzMuNXiz4/Jc',
    '8Mp4AeyodykO13/97fT6RRDemIOTMRHWLQgrVr2xbLxolmsFf48SbX1jqESuammWjEeT0J9cFKBYFKXcN0mkB4ZEcm8CcOMn',
    'STqI1yQnWYzzYTbA/2dUC4DDeBELMoXpolr4NBFILS1tMk4DMLa39wcP1PQUfJ7R2Bn4CpWU9YyGFUEBuqB95hUfgXRaT8Sy',
    'b1ywQ8AOP/Hv8vwsWfsj2NuGNHQT/oUJWDKN3z58OIECb2JoyTzDJSu54klHEQRtuE3duFoPA7ATJHMW5NaxJae5ak24TXZh',
    'wFAjMYi4d+CUziY7ZIq2gKytsc85A2KRCb9c1MrheHdvLd04IPUtShSNgeLx/b7ZHbyOcLbbT53Umevf/9yi9j3HdsbSn/d3',
    'k+ge4G523bnsQIPJUkGJeE4F8NiilVQ5gqX5S7ZH9+86npBO+EBHGAAJBOTrWBy6RZUjfYtT2OoW0qyEJGdOkcluA8m1poO1',
    'bxGtQq3pZkrVIJPV/imaOShrRksKGDnjpPgZaa1F1T+0UrsdmQTaxuidNNSmQoh8yrP3WLHhGCri9vIMZvyOvdqDURDT1iQZ',
    'yk/l2iJLYCe8SC8kUD/tFqJbIH+0Wtd+wwcB9iCvoMQI/PMFyZO5k6vKYzV8Riyh4SQsOHUuDGGL0iJ7VeUtgjN+ROTgdVRJ',
    'zSwJN+9ASsQTlhYsbfDJSpkj5XC1WyIzoRHtEX3KAOyKpD+PeXWgsFmV9Y2PCeY36FsfH4/Hq4+PGYIhRaqxz8HpTDSbvf8A',
    'UEsDBBQAAAAIAMgzQ11f02OWdAcAAEgPAAAaAAAATWF0aC9wcm9vZnMvcHJvb2ZfdGhtMDkubWR9V92O2zYWvvdTEOhF7LHl',
    'yXg7i25aXaRtigbNYoPtdFtgkEg0RY85lUSVpMZ247lvX2Dvi7YvEGCBXOcB5iHyJPsdkvqZtOiVZYo6PD/f953DD9jFl/9M',
    'Hv7jEZN7LhzbqL0sEiOv2pIb5Q5M1U6aG14mqt5oU3GndM0ao4Rk+M90LZluXdM6VkihLN5OJicnXzvuWvuIPf/3v/7z5HO/',
    '020ls1iWBdvwSpUHxutiMK9rrGykLNZcfL88OWHPpGP5d+nlw8XZi3zBcpGena7oweDhIzzQ5/nDT+y7n3+jhSV7zLaqKCTc',
    '44ZXEoZZfvfm3U8/XYrELsTcvsiZsiFE7xIvS5Zf5Mzoti7skl3ARbfTrISXtWN6fS2FUzfSMm7kZJJvs7OXd2+m+1k63Sd3',
    'b+Zm9tJ7tM1W4/XEry8nk8eOyRtpDuEAn4FSclPDMSth1acEnjKhq6aEw3062KatBaXaekeDFQSTP4Ibz7Jrf9Zxn4js+pjY',
    'WTZ/+5oc+fL+m7md0fpksttKI5HC7CwViclD6kS2SsXc5D5uKylG71Gj4cVOYaVStaraym+v+N4/6w3Fex3izZmGZ+EraSrl',
    'qLpD9rto/AkHf4DQ9Y3cL5g2BVwqQhWvkGOGfaqgTFQA2JCHiIgFlkTZFqq+8mWTdeH9ZFeGFwrVsqe2Xfd/Qi07n4XGcapG',
    'VdlOFW5LKMi/Tafnp6uZzT+G/y0Wvsku4tIFcvKpxr4/BQJbJdYZXV8BsCEcHwR3rNLW4e0z1VixVe5HsAMQzsn3QjZwGcbw',
    'EaXQEj48xO/+lyGXr/a3yOY6VMBT8pzttLEucdxcYd9znOx0YvEoKOn8CmDS1pI1zizyUvZM9JmJBIH5s5cn07s3sxQMzl7t',
    'wYfvbkeH+lTV/twgAb7qfI9/DaJGQU9O5F5Ia0FKZYE/svOY2bbJXt2n1y17ghd3v2aOXXYnZO52lgxOvGApu5iujJ3bt69n',
    '+DM9Oz17OF/Z0/MZQwUoLU89g50SpEFInrPMkO4wp5loK6xSLQaMhJqWCuVZS7eTiCYnqxHnfztd4XG3VaWMIkTFS0LpSOJo',
    '11CztRcDZmTFVR3EIqIp5oWXVxrSuK1iugEdRJmKfETVQHgyXAPAltW6RzL7ocWOYLLUO9BkHTaX+EcAWytnONnA57pSP8r3',
    'VbJ3wPoTcGI9KEgDDCadFPc0STo96d0g1EsFDYwyE+MdIZWQRfmKokrqsyRtf2603pBCd3LZ0yOkCB6bQJSctDBWgeQSTLA6',
    'IG2zUYKYygjQoKOHHmLLL+mbBe1+AWh+QSod0e2ITYAsrPgytg18JkUiCRoMBr7Y+6JFWwYVsF5mWieX7AkX21EA3DlkwfbK',
    'h1A0MZswqHf1ECASBS/wZji4K9KCgU+C+sXAshSevOqaxtvXi65PvH19m07N/Eh/jyTWBH+c1dYKMBkfRyhLUYIFqSWp4JjZ',
    'hr709fiTBsLXCB/YJHyx9aEz6pW0y1H469P6Q4vSFaxQaNY1dXmjK5ydE/vc+IT8Ej2GmJ9dU2sdI1qCWqVuJLn9B4Owwzvh',
    '7I3x2B/bpnnvezcSckIkANeLP0wXJA3CemStSbOHLrNk34ImPleHNLbEAB7gPoiGhIUSqf3QHgJ988O7n3+3EbPTQ+ig/Zvf',
    'INvsa+WjGLkFzlm1hrzksEIoDUOLLxqGlrlJz4eJpZtXFvdMDH3pQ4sm9NEsPbcYdzDUlAgsP05Dz509QNc9wsJqfFC03O95',
    'kPrx44s4dY1VxrcHo/Cm4ja45y1Jz4MN8TRLMCTYnrbZHFMCxf2UREEJFOsP3RnGvger4pgCNczfFxLILw7oC4fWVhZsx8HE',
    'rbayXnQqVheKUAvbOvSkXuweBAY2HD22k8YwjZKjem3h0kgqbRCP4FIY+XLf+2LinVHc90xVexBQI6A5hBrc1PMxS46BmNn8',
    'ODtdMQCDWXwdOkk+nYZNAMg8PM7xiI00BApaRYcjMAVWX9AIBhzFGYtKkFSS1zRBJ6Fd9H0dMICyYFREqDm4PY/9ksDXrtEd',
    'hOvoGucljTU4tSk1IdX4adC2VUXbvPhhzA0o4f6Mfn1oZiGnNfo8s1vqRF7eej9o0A9pfK+IW26HUWHUnsPMMApkaPuI4zNi',
    'qLGyPIRyBNNxfun6KZnOj/hD6ujZN2qxw8AZ/IgKEpM8nO6HsrzLYAC6zzcqqToJ8GOAC63pXnh+kkQ9GqNv4sR+f0pSnhXu',
    '4Etj8dQGxQm3lEAimizP7QVRzUMsRkwo94PNkr377y/+2iQgeqG1KmrJDVdopX4c/Dvwz9HjcUAhEwsa0jEC5YTfXp++Ss/y',
    'R1FIMLZi6oDS+gY2nn38BWx0vwPlSJfRc/Bz6Oadp7jOaUlzCzocXqxLZbekFl/lrDubx9vi2Br8IPBE7ITbThjZO5ipKqYy',
    '8rsfbfiOlDtYFJJWP/7LexH6icAAHOIKH+BOk3vfwzBjQ5/BvNd6cBLTClXJ2gaFCddQ3zlquFTGUXBtNC8SUZI+8gYZF3yt',
    'PEIMsKLgpJV0x4Fb8obKL+Ry8n9QSwMEFAAAAAgArzVDXZe4AOR/BgAA+w0AABUAAABNYXRoL3RoZW9yZW1faW5kZXgubWSN',
    'V0ty20YQ3esUU5WNJBOSKP+L5QUtMTFL1CcS5bgqlRKGwBCceDCDzAxIyaVFssw6mxwgJ/ARfIDkDj5JXg9AgLIoJwuVAALT',
    'n9evXze+YeOZMFbkTOpUXG9s3LLhIbtl58KVyuPiwnNfOlz0nSvzwkuj6e5QFAIHdCIF3Z6YuVD+hsm8MNZznQh2u3EbRdHa',
    'PzgZvzmO9ro4Obpk0jHYKozUnhXWpGXimbGpsHh8dn76dkABveVKpgjSCzvnqgphKrVcBjQyix6bmlKnnH7iijVu9unlG81z',
    'mbBc1o5OvztkE3qdLaSfMb8wkTJJOItfUj9zq9773c+//tF/Sj9Z87NI6LUOS4yei2vmyklmeSqF9uE3z6XOcYO3jw3S4F4w',
    '7hhnE+6Eklr0mJLvhbphhbC+tJPKK7dZWR1bBv4YFoZ1xpHRODAVIp3w5D0ytQmQjy+vxrv7MRPXBYISKbMis8IDJ7ijnBbG',
    'KoLN1UVZSelJp4HzrvEOw9tkbOXYUKfSeamzUroZn0glUW1YMx4RS65wPMGV5aqD5DwlXVrB5sLKqaxh1bAPq016T2DgVAOb',
    'ACe8JaVFSiGhUkukmEeFFVNhBQVRpxbKJa5xiLCeKplQWMxMQlnm4k7dLu4l0leZmFhOFC8VXjalZ1wz8Nc6bw0sJYrLvI2S',
    'iv4tEU1EkzLNEMEZRxwmcoIiwDmeiZpKU2tyVnCgoIRquaYEtxr27zKqw/qPN1vic+u3OqEq/edEtu4ea/jLYXsZXeDcV9Cf',
    'UEJNr0Yzwec3IZvRoG65QQCve8h+KXmKQqEtqpQaPgSHCyGzmY9yXrCRLFwyk/4DuSco/RqQW2tTtJqiAA9MXigR6uPw2IoO',
    'y4023miZVCmEtsZxx9yN9jNBx2vY26Ih+P7ZWRX8hcwhTFwLA1VCNKms270wRakqok0QvqMWYYN3ZxGAVCYj9ia4cC4kh5pF',
    'LoHwsQz5rSTzA+gldYSeTGZsCC1c9QHbBXe+V5UbOQNy7YiouJ6KwHnXQyhOEhVZZk1ZkKZNqpapxPKNEdNpClA6LCgYvSfT',
    'DnGegg9MQsZFoZadM8HzDLLBVzQYMNbtBd945/JgyJIZOGKQ7U3L32ewFWRi88Xe0Vbc9rzUFHjlQJkF5LYmMXDb3hbX0Be3',
    'vb2W7UFeVrtvIpv+W8eDCt0OWhgslkGo0GUOUGnDcoKajGQCTQKUg2DeU75SqXZMtFIVH8WMYkMfF6V3BOk1zM9kiunEKvlr',
    'OrlJIBcUx/oWQv2sNDZCQ97VL5p4FGcD7XNS+GX08d9/bhLMhHGOsub8mlUgtm6rqUYAg+rL8vwnSsR6nocZUh3pod4iSiWm',
    'hauImZocQ6fH4n0eM3QBjZG2aZeUqh0G/i8VtlUpImHoZshthvjAsRqRlYiiOtogkr2mZE0NQ+0MkGpRekECWtcrmlqxRD/S',
    'KBtKF30hovHon9+ujjbfbY3jFVCWntqsMmHI0FJnSNPidzFNYMAsNQJZSCeWaMSjOGrPttPiHoO+xpbX3OF/7RhX7UBPMENQ',
    'hOCWYY5Z8lnbyaB8kEwhWkxeNjIcH73qxhVpI0w4SJglZfyCQfF4c9+6R+7Tx61Xm93d7t6jfbf7dAt8i9cTJwiTeliXw+LS',
    'yL3ArULV1rbPvTXhgU4EJ2sMWfXoISB7dzaTtXvFucgNjfIlYzFGpC9T0mwMDBqDfA65Da0UEGypSOUkhut6JW3IGObffgM8',
    'xVDF2cpaSlqAM3TeWJ4owabKkI+X9tPH3e6zB9Cua9rg+9pgRwnGHWkl5B7zjhixRBKEUbIoKJMZnIVFpPJ3y75v6tUExpul',
    'ZWyMClnfYwxg2KfMubpx2KgpY0R6VqVMIxoGm0LyBVS9PlJTrCywjYJoP4Jpsyv3CH9gW1Qn/tP/zDx2n3//y+6+QB8C4FaV',
    'ofOYij5KRSJJs8BKSGsYh01dLkMExLEedERTO32Ai2UrVDOqKefw5G21EQwDlylxk5Wi2g/RjdVWySdUQ4ndLqrEt4H5ZDA4',
    'vGD9w8PheHh60h+x/sXF5fEZ3TRGafdJSFu8JDvBdFnQvrKqPvWShjFvpljTBO0TF4L2Bk7PNzZOTNiosbDMsMvS9w5v6BmE',
    'dIf1V7dmXmLjoNcAH+1ANHXwzYDPE8U0vhGAPa7m1eoUVmeHmiSUIHUUi3d2dg/w5bE7fjM4PR8cX51cHg/Ohwf90VX/Evnu',
    '5Gm8s/EvUEsBAhQAFAAAAAgAkTVDXesRLkI7BAAALgoAAC4AAAAAAAAAAAAAALaBAAAAAENvZGUvZXhwZXJpbWVudHMvY2hl',
    'Y2tfazJfY292ZXJhZ2VfaWRlbnRpdHkucHlQSwECFAAUAAAACAAINUNdSPMsBLYCAADBBQAAJAAAAAAAAAAAAAAAtoGHBAAA',
    'Q29kZS9leHBlcmltZW50cy9leHBsb3JlX2ZpeGVkX2syLnB5UEsBAhQAFAAAAAgAfTVDXW+SUf7aBgAAYQ0AACkAAAAAAAAA',
    'AAAAALaBfwcAAE1hdGgvcHJvb2ZzL3Byb29mX2syX2NvdmVyYWdlX2lkZW50aXR5Lm1kUEsBAhQAFAAAAAgAyDNDXV/TY5Z0',
    'BwAASA8AABoAAAAAAAAAAAAAALaBoA4AAE1hdGgvcHJvb2ZzL3Byb29mX3RobTA5Lm1kUEsBAhQAFAAAAAgArzVDXZe4AOR/',
    'BgAA+w0AABUAAAAAAAAAAAAAALaBTBYAAE1hdGgvdGhlb3JlbV9pbmRleC5tZFBLBQYAAAAABQAFAJABAAD+HAAAAAA='
]))
(RUN_ROOT / "source_package.zip").write_bytes(SOURCE_ARCHIVE)
with zipfile.ZipFile(RUN_ROOT / "source_package.zip") as archive:
    for member in archive.infolist():
        target = (RUN_ROOT / member.filename).resolve()
        if not target.is_relative_to(RUN_ROOT.resolve()):
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(RUN_ROOT)
record("sources", "passed", "K=2 checks, proof, manuscript source and PDF")


## 3. Check five K=2 identity and upper-bound settings


In [ ]:
if RUN_OK:
    started = time.perf_counter()
    command = [sys.executable, str(RUN_ROOT / "Code" / "experiments" / "check_k2_coverage_identity.py")]
    result = subprocess.run(command, cwd=RUN_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout[-5000:])
    if result.returncode:
        RUN_OK = False
        record("experiment", "failed", f"exit={result.returncode}")
    else:
        summary_path = RUN_ROOT / "Code" / "results" / "fixed_k2_coverage" / "fixed_k2_coverage_summary.json"
        summary = json.loads(summary_path.read_text())
        RUN_OK = bool(summary["cases"] == 5 and summary["all_checks_passed"]
                      and summary["max_identity_error"] < 1e-10)
        record("experiment", "passed" if RUN_OK else "failed",
               f"five cases; seconds={time.perf_counter()-started:.2f}")


## 4. Zip outputs and sources


In [ ]:
(RUN_ROOT / "notebook_stages.json").write_text(
    json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
FINAL_ZIP = WORK_ROOT / "result_jcam_k2_fixed_regularity_upper.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file() and path.name != "source_package.zip":
            archive.write(path, arcname=(Path("k2_fixed_regularity_upper") / path.relative_to(RUN_ROOT)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
